In [1]:
%load_ext autoreload
%autoreload 2

import sys
import json
sys.path.append("..")

from datasets import load_dataset
from src.donut import load_donut, predict
from src.utils import FIELDS, parse_ground_truth, get_field, normalize_number

val = load_dataset("naver-clova-ix/cord-v2", cache_dir="../data", split="validation")
processor, model = load_donut(cache_dir="../data/hf_models")

preprocessor_config.json:   0%|          | 0.00/362 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/4.74k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/536 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/4.02M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/1.52k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/335 [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  806MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/484 [00:00<?, ?it/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  806MB            

model.safetensors: downloading bytes:           |  0.00B            

In [2]:
for i in range(5):
    parsed, sequence, seconds = predict(processor, model, val[i]["image"])
    gt = parse_ground_truth(val[i]["ground_truth"])

    print(f"--- Zettel {i}  ({seconds:.1f} s)")
    for section, key in FIELDS:
        g = normalize_number(get_field(gt, section, key))
        p = normalize_number(get_field(parsed, section, key))
        mark = "✓" if g == p else "✗"
        print(f"  {mark} {key:15s} gt={g}  pred={p}")

--- Zettel 0  (4.4 s)
  ✓ total_price     gt=45500  pred=45500
  ✓ subtotal_price  gt=None  pred=None
  ✓ cashprice       gt=50000  pred=50000
  ✓ changeprice     gt=4500  pred=4500
  ✓ tax_price       gt=None  pred=None
--- Zettel 1  (2.7 s)
  ✓ total_price     gt=23000  pred=23000
  ✓ subtotal_price  gt=None  pred=None
  ✓ cashprice       gt=50000  pred=50000
  ✓ changeprice     gt=27000  pred=27000
  ✓ tax_price       gt=None  pred=None
--- Zettel 2  (3.1 s)
  ✓ total_price     gt=20000  pred=20000
  ✓ subtotal_price  gt=18181  pred=18181
  ✓ cashprice       gt=100000  pred=100000
  ✓ changeprice     gt=80000  pred=80000
  ✓ tax_price       gt=1818  pred=1818
--- Zettel 3  (3.3 s)
  ✓ total_price     gt=28000  pred=28000
  ✓ subtotal_price  gt=28000  pred=28000
  ✓ cashprice       gt=28000  pred=28000
  ✓ changeprice     gt=0  pred=0
  ✓ tax_price       gt=0  pred=0
--- Zettel 4  (2.9 s)
  ✓ total_price     gt=41000  pred=41000
  ✓ subtotal_price  gt=41000  pred=41000
  ✓ cashprice 

In [3]:
parsed, sequence, seconds = predict(processor, model, val[2]["image"])
print(sequence[:500])
print()
print(json.dumps(parsed, indent=2, ensure_ascii=False))

<s_menu><s_nm> S-Ovaltine</s_nm><s_unitprice> 20,000</s_unitprice><s_cnt> 1</s_cnt><s_price> 20,000</s_price><s_sub><s_nm> 50%</s_nm></s_sub></s_menu><s_sub_total><s_subtotal_price> 18,181</s_subtotal_price><s_tax_price> 1,818</s_tax_price></s_sub_total><s_total><s_total_price> 20,000</s_total_price><s_cashprice> 100,000</s_cashprice><s_changeprice> 80,000</s_changeprice></s_total>

{
  "menu": {
    "nm": "S-Ovaltine",
    "unitprice": "20,000",
    "cnt": "1",
    "price": "20,000",
    "sub": {
      "nm": "50%"
    }
  },
  "sub_total": {
    "subtotal_price": "18,181",
    "tax_price": "1,818"
  },
  "total": {
    "total_price": "20,000",
    "cashprice": "100,000",
    "changeprice": "80,000"
  }
}
